# 002 Streaming Chat Agent Flow

这是 Harness Engineering 教学目录的第二份 Notebook。

学习目标：

1. 理解当前项目里的流式问答智能体完整链路
2. 看懂系统如何控制 AI：先规划、再裁决权限、再执行工具、最后流式回答
3. 理解前端 `/chat` 如何消费 `text/event-stream`
4. 理解为什么 Harness 不是“调用一次模型”，而是一条受控执行流水线

这份 Notebook 直接对应当前项目实现，不是抽象概念。

## 你要先记住的一句话

这个流式聊天智能体不是这样工作的：

```text
用户问题 -> 大模型 -> 回答
```

而是这样工作的：

```text
用户问题
  -> FastAPI Router
  -> Service
  -> Harness Query Loop
  -> Skill 选择
  -> 模型规划 plan
  -> Tool 权限裁决
  -> Tool 执行
  -> 模型流式组织回答
  -> SSE 分块返回前端
```

AI 只负责其中的两个环节：生成计划、组织回答。其它执行秩序由系统控制。

## 本项目对应文件

先建立文件地图：

| 层级 | 文件 | 作用 |
|---|---|---|
| 页面 | `app/static/chat/index.html` | 聊天 UI，读取 SSE 流 |
| Router | `app/api/routers/chat_agent.py` | 暴露 `/chat/stream` |
| Service | `app/services/chat_agent.py` | 调用 Agent Runtime |
| Harness | `app/agents/harness.py` | Query Loop、ledger、权限、工具、回答 |
| Model Client | `app/agents/model_client.py` | OpenAI 兼容模型调用和流式调用 |
| Skill Registry | `app/agents/skills.py` | 加载和安装 Skill |
| Tool Registry | `app/agents/tools.py` | 注册、权限裁决、执行 Tool |
| Schema | `app/schemas/chat_agent.py` | 请求响应 DTO |

学习时建议按这个顺序读代码，不要从模型调用开始读。

## 学习计划

建议按 5 个阶段学习：

1. 先跑通页面：打开 `/chat`，发一条普通问题和一条天气问题
2. 再看网络流：观察 `ledger`、`plan`、`tool_result`、`answer_delta`、`done`
3. 再读后端入口：从 `chat_agent.py` 的 `/chat/stream` 开始
4. 再读核心循环：重点看 `HarnessChatAgent.stream_chat(...)`
5. 最后读模型和工具：看 `_plan(...)`、`_answer_stream(...)`、`ToolRegistry.execute(...)`

这个顺序符合真实请求路径，也最不容易迷路。

## 第一步：前端不是 EventSource，而是 fetch 读取 POST 流

浏览器原生 `EventSource` 适合 GET SSE。

当前接口是 POST，因为我们要提交 JSON：

- `message`
- `session_id`
- `skill_names`
- `auto_approve_tools`
- `max_steps`

所以页面使用的是：

```javascript
fetch('/api/v1/chat-agent/chat/stream', {
  method: 'POST',
  headers: {'Content-Type': 'application/json'},
  body: JSON.stringify(payload)
})
```

然后通过 `response.body.getReader()` 一段一段读取 SSE 文本。

## 第二步：Router 把 Agent 事件包装成 SSE

`app/api/routers/chat_agent.py` 里关键代码是：

```python
@router.post('/chat/stream')
def stream_chat_with_agent(payload: ChatAgentMessageRequest):
    return StreamingResponse(
        _sse_events(chat_agent_service.stream_chat(payload)),
        media_type='text/event-stream',
    )
```

`chat_agent_service.stream_chat(...)` 产出的是 Python dict：

```python
{'event': 'answer_delta', 'data': {'text': '你好'}}
```

`_sse_events(...)` 负责转成 SSE 文本格式：

```text
event: answer_delta
data: {"text":"你好"}

```

In [1]:
import json


def to_sse(event: str, data: dict) -> str:
    return f"event: {event}\ndata: {json.dumps(data, ensure_ascii=False)}\n\n"


print(to_sse("answer_delta", {"text": "你好"}))

event: answer_delta
data: {"text": "你好"}




## 第三步：Service 不做复杂逻辑，只做边界转发

`app/services/chat_agent.py` 的职责很薄：

```python
def stream_chat(self, payload):
    yield from self.agent.stream_chat(...)
```

这符合当前项目的分层习惯：

- Router 管 HTTP
- Service 管业务入口
- Agent Runtime 管智能体流程

不要把 Query Loop 写进 Router。Router 应该保持薄。

## 第四步：Harness Query Loop 是核心

`HarnessChatAgent.stream_chat(...)` 是整套系统的心跳。

它按固定顺序推进：

1. 创建或读取 `session_id`
2. 读取会话历史
3. 根据用户消息选择 Skill
4. 裁剪历史上下文
5. 调模型生成 `plan`
6. 如果 plan 要调用工具，先做权限裁决
7. 执行工具并记录结果
8. 调模型流式生成最终回答
9. 更新会话历史
10. 返回 `done` 事件

这就是 Harness 控制 AI 的地方：模型不能跳过权限，也不能直接执行工具。

## 第五步：ledger 是系统的执行账本

每个关键阶段都会写入 ledger。

典型顺序：

```text
input
context
plan
permission-1
tool-1
tool-result-1
answer
done
```

这让系统具备可解释性：你能看到它为什么调用工具、是否经过权限裁决、工具结果是什么。

In [2]:
sample_ledger = [
    {"step": "input", "state": "ready", "detail": "收到用户消息"},
    {"step": "context", "state": "planning", "detail": "完成上下文裁剪"},
    {"step": "plan", "state": "planning", "detail": "模型生成行动计划"},
    {"step": "permission-1", "state": "tool_permission", "detail": "完成工具权限裁决"},
    {"step": "tool-1", "state": "tool_running", "detail": "开始执行工具"},
    {"step": "tool-result-1", "state": "tool_running", "detail": "工具执行完成"},
    {"step": "answer", "state": "answering", "detail": "完成最终回答"},
    {"step": "done", "state": "done", "detail": "会话状态已更新"},
]

[item["step"] for item in sample_ledger]

['input',
 'context',
 'plan',
 'permission-1',
 'tool-1',
 'tool-result-1',
 'answer',
 'done']

## 第六步：系统和 AI 有两次不同交互

这一点很重要。

当前系统不是只调用一次大模型，而是至少分成两个模型阶段：

### 1. Planning 调用

`_plan(...)` 调用：

```python
self.model_client.chat([...])
```

它要求模型只输出 JSON：

```json
{
  "action": "tool",
  "tool_name": "weather.current",
  "arguments": {"location": "北京"},
  "reason": "用户询问实时天气"
}
```

### 2. Answer Streaming 调用

`_answer_stream(...)` 调用：

```python
self.model_client.stream_chat([...])
```

它把用户问题、plan、tool_result 给模型，让模型分片生成最终回答。

## 第七步：Skill 不执行，Skill 影响 Prompt

当前 Skill 的作用不是直接运行代码。

`SkillRegistry.select_skills(...)` 会根据消息选择相关 Skill，然后 Harness 把 Skill 摘要拼进模型提示词。

也就是说：

- Skill 解决“应该怎么做”
- Tool 解决“具体能做什么”

例如天气问题：

- 选中 `weather` Skill
- 模型看到天气 Skill 的规则
- 模型规划调用 `weather.current` Tool
- 系统做权限裁决并执行 Tool

这就是 Skill 和 Tool 分离。

## 第八步：Tool 调用前必须经过权限裁决

`ToolRegistry.decide_permission(...)` 会返回：

- `allow`：允许执行
- `ask`：需要用户批准
- `deny`：拒绝执行

当前天气工具是低风险工具，所以默认允许。

如果以后接入 Bash、文件写入、数据库修改，就不能默认允许。必须提高 risk level，让 Harness 阻断或要求审批。

In [3]:
def decide_permission(risk_level: str, auto_approve: bool = False) -> str:
    if risk_level == "low":
        return "allow"
    if risk_level == "medium":
        return "allow" if auto_approve else "ask"
    return "ask"


for risk in ["low", "medium", "high"]:
    print(risk, "=>", decide_permission(risk))

low => allow
medium => ask
high => ask


## 第九步：为什么工具结果先返回，再流式回答

流式接口不是只流模型 token。

它先流出执行过程：

```text
ledger
plan
permission
tool_start
tool_result
```

然后才流：

```text
answer_delta
answer_delta
answer_delta
done
```

这就是 Harness 风格的可解释流式输出：前端不仅看到答案，还能看到答案是怎么来的。

## 第十步：前端如何理解 SSE 事件

页面里的核心逻辑是：

- `answer_delta`：追加到当前助手气泡
- `done`：保存 `session_id`
- 其它事件：显示到右侧执行事件面板

这让用户同时获得两种视图：

1. 左侧：正常聊天体验
2. 右侧：系统执行账本

这比普通聊天窗口更适合学习和调试智能体。

In [4]:
def handle_event(event_name: str, data: dict, answer: str) -> str:
    if event_name == "answer_delta":
        return answer + data.get("text", "")
    if event_name == "done":
        return answer
    return answer


answer = ""
for event_name, data in [
    ("answer_delta", {"text": "北京"}),
    ("answer_delta", {"text": "天气"}),
    ("answer_delta", {"text": "晴朗"}),
]:
    answer = handle_event(event_name, data, answer)

answer

'北京天气晴朗'

## 第十一步：用 FakeModelClient 本地观察完整事件

下面这个例子不调用真实大模型。

它用假的模型客户端模拟两件事：

1. 第一次模型调用返回 plan
2. 第二次模型调用流式返回 answer_delta

这样你可以专注理解 Harness 流程，而不是被模型网络调用干扰。

In [5]:
import tempfile
from pathlib import Path

from app.agents.harness import HarnessChatAgent
from app.agents.skills import SkillRegistry
from app.agents.tools import ToolRegistry


class FakeModelClient:
    def __init__(self):
        self.calls = 0

    def chat(self, messages, model=None):
        self.calls += 1
        return '{"action":"tool","tool_name":"echo","arguments":{"text":"hello"},"reason":"演示工具调用"}'

    def stream_chat(self, messages, model=None):
        for chunk in ["工具", "返回", " hello"]:
            yield chunk


temp_dir = tempfile.TemporaryDirectory()
agent = HarnessChatAgent(
    model_client=FakeModelClient(),
    skill_registry=SkillRegistry(installed_dir=Path(temp_dir.name)),
    tool_registry=ToolRegistry(),
)

events = list(agent.stream_chat("请 echo hello"))
[(event["event"], event["data"].get("step") or event["data"].get("text") or event["data"].get("tool_name")) for event in events]

[('ledger', 'input'),
 ('ledger', 'context'),
 ('ledger', 'plan'),
 ('plan', 'echo'),
 ('ledger', 'permission-1'),
 ('permission', 'echo'),
 ('ledger', 'tool-1'),
 ('tool_start', 'echo'),
 ('ledger', 'tool-result-1'),
 ('tool_result', 'echo'),
 ('answer_delta', '工具'),
 ('answer_delta', '返回'),
 ('answer_delta', ' hello'),
 ('ledger', 'answer'),
 ('ledger', 'done'),
 ('done', None)]

## 第十二步：真实天气问题的流程

当你在页面输入：

```text
北京现在天气怎么样？
```

理想流程是：

1. 前端 POST 到 `/api/v1/chat-agent/chat/stream`
2. Harness 选中 `weather` Skill
3. 模型 plan 返回 `weather.current`
4. ToolRegistry 判断 `weather.current` 是 low risk，允许执行
5. Tool 调用 `.agents/tools/weather/fetch_weather.py`
6. 工具结果通过 `tool_result` 事件流给前端
7. 模型根据工具结果流式生成中文回答
8. Harness 保存 session 历史
9. `done` 事件返回完整结果

这条链路体现了 Harness Engineering 的核心：AI 负责规划和表达，系统负责边界和执行。

## 第十三步：会话记忆是怎么工作的

当前版本的会话历史保存在内存里：

```python
self._sessions: dict[str, list[dict[str, str]]] = {}
```

每次请求：

1. 前端带上 `session_id`
2. Harness 读取历史消息
3. `_compact_history(...)` 只保留最近 N 条
4. 回答完成后追加 user / assistant 消息

这就是最小版上下文治理。

后续如果要生产化，可以把 session 存到 Redis 或数据库，并增加更强的摘要压缩策略。

## 第十四步：这个版本的控制边界

当前版本已经具备：

- 模型客户端隔离
- Skill 和 Tool 分离
- Tool 权限裁决
- SSE 流式输出
- ledger 执行账本
- session 历史
- fake model 测试能力

当前版本还没有做：

- 真正的中断恢复
- 长任务取消
- 高风险工具审批回放
- 多工具多步循环
- 持久化会话记忆
- 独立 verifier agent

这很正常。第一版先把受控链路跑通，再逐步增强。

## 调试清单

如果流式聊天不符合预期，按这个顺序排查：

1. 页面是否能打开：`GET /chat`
2. 流式接口是否返回 SSE：`POST /api/v1/chat-agent/chat/stream`
3. 是否有 `plan` 事件
4. plan 是否选择了正确工具
5. 是否出现 `permission` 事件
6. 是否出现 `tool_result` 事件
7. `tool_result.ok` 是否为 true
8. 是否持续出现 `answer_delta`
9. 最后是否出现 `done`
10. `done.ledger` 是否完整

这比只看最终回答可靠得多。

## 关键理解：系统如何控制 AI

可以把控制点总结成 7 个：

1. **Prompt 控制**：规划阶段要求模型只输出 JSON
2. **Skill 控制**：只把选中的 Skill 摘要喂给模型
3. **Tool 控制**：模型只能选择注册过的工具名
4. **Permission 控制**：工具执行前必须经过 allow / ask / deny
5. **Context 控制**：历史消息会裁剪，不无限增长
6. **Stream 控制**：系统事件和回答 token 分开流
7. **Ledger 控制**：每一步都留账本，方便解释和调试

所以，模型并没有直接控制系统。模型只是在 Harness 允许的边界里提出计划和组织语言。

## 本阶段小结

这份 Notebook 要你掌握的不是某个 API 用法，而是一种智能体工程结构：

```text
HTTP 请求
  -> 受控 Query Loop
  -> 模型规划
  -> 权限裁决
  -> 工具执行
  -> 模型流式回答
  -> SSE 事件
  -> 前端实时渲染
```

后续最自然的学习方向是：

1. 给 Tool 增加 medium / high risk 示例
2. 增加审批确认接口
3. 增加多步工具循环
4. 增加中断与恢复
5. 把 session 历史持久化

这些都是 Harness Engineering 从 demo 走向工程系统的关键步骤。